# Simons on GIFT-Eval

Zero-shot evaluation of Simons on the full GIFT-Eval benchmark (97 configurations: 55 short-term,
21 medium-term and 21 long-term), following the official protocol of the GIFT-Eval notebooks.

- Model: `abel-lab/Simons-297M` (or `-178M`, `-60M`, `-25M`); set `MODEL` below.
- Forecasts: nine quantiles (0.1 ... 0.9) per step; the mean forecast is the mean of the quantiles.
- Multivariate datasets are forecast jointly (at most 32 variates attend to each other).
- Setup, with Python 3.10 to 3.12 (gift-eval pins `numpy~=1.26`), at gift-eval commit `82f89c3`:

  ```shell
  git clone https://github.com/SalesforceAIResearch/gift-eval && git -C gift-eval checkout 82f89c3
  pip install -e gift-eval "simons @ git+https://github.com/Abel-ai-lab/Simons" "pyarrow<21"
  hf download Salesforce/GiftEval --repo-type=dataset --local-dir gift-eval-data
  export GIFT_EVAL=$PWD/gift-eval-data GIFT_EVAL_REPO=$PWD/gift-eval
  ```

  `GIFT_EVAL` is the data folder; `GIFT_EVAL_REPO` is the gift-eval checkout (default: the current
  folder), whose `notebooks/dataset_properties.json` and Seasonal Naive results the notebook reads.
- On a GPU the transformer runs in bfloat16 (embedding, normalisation and output head in float32).
- Output: `results/<model>/all_results.csv` and `results/<model>/config.json`, the two files of a
  [submission](https://github.com/SalesforceAIResearch/gift-eval#submitting-your-results).

In [ ]:
import csv
import json
import os
from pathlib import Path

import torch
from gift_eval.data import Dataset
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality

from simons import Simons
from simons.predictor import SimonsPredictor

MODEL = os.getenv("SIMONS_MODEL", "abel-lab/Simons-297M")
NAME = os.getenv("SIMONS_NAME", MODEL.split("/")[-1])
REPO = Path(os.getenv("GIFT_EVAL_REPO", "."))  # the gift-eval checkout
PROPERTIES = os.getenv("GIFT_EVAL_PROPERTIES", REPO / "notebooks/dataset_properties.json")
OUTPUT = Path(os.getenv("SIMONS_OUTPUT", f"results/{NAME}"))

device = "cuda" if torch.cuda.is_available() else "cpu"
model = Simons.from_pretrained(MODEL, device=device)
print(f"{NAME}: {sum(p.numel() for p in model.parameters()):,} parameters on {device}")

In [ ]:
# GIFT-Eval's submission metadata, next to the results (see the gift-eval README).
OUTPUT.mkdir(parents=True, exist_ok=True)
submission = {
    "model": NAME,
    # zero-shot: the training data holds no GIFT-Eval test data and no GIFT-Eval train split
    "model_type": "zero-shot",
    "model_dtype": "bfloat16" if device == "cuda" else "float32",  # the transformer's precision
    "model_link": f"https://huggingface.co/{MODEL}",
    "code_link": "https://github.com/SalesforceAIResearch/gift-eval/blob/main/notebooks/simons.ipynb",
    "org": "Abel AI Lab",
    "testdata_leakage": "No",
    "replication_code_available": "Yes",
}
(OUTPUT / "config.json").write_text(json.dumps(submission, indent=4) + "\n")
print(json.dumps(submission, indent=4))

In [ ]:
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]

short_datasets = (
    "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly "
    "electricity/15T electricity/H electricity/D electricity/W "
    "solar/10T solar/H solar/D solar/W hospital covid_deaths "
    "us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W "
    "temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D "
    "car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W "
    "LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H "
    "M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W "
    "ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D "
    "bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H "
    "bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
).split()
med_long_datasets = (
    "electricity/15T electricity/H solar/10T solar/H "
    "kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H "
    "ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H "
    "bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service "
    "bizitobs_l2c/5T bizitobs_l2c/H"
).split()
all_datasets = sorted(set(short_datasets + med_long_datasets))

properties = json.loads(Path(PROPERTIES).read_text())
pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT / "all_results.csv"
header = [
    "dataset", "model",
    "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]",
    "eval_metrics/MASE[0.5]", "eval_metrics/MAPE[0.5]", "eval_metrics/sMAPE[0.5]",
    "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]",
    "eval_metrics/ND[0.5]", "eval_metrics/mean_weighted_sum_quantile_loss",
    "domain", "num_variates",
]
with open(csv_path, "w", newline="") as stream:
    csv.writer(stream).writerow(header)

for ds_name in all_datasets:
    for term in ("short", "medium", "long"):
        if term != "short" and ds_name not in med_long_datasets:
            continue
        if "/" in ds_name:
            key, freq = ds_name.split("/")
            key = pretty_names.get(key.lower(), key.lower())
        else:
            key = pretty_names.get(ds_name.lower(), ds_name.lower())
            freq = properties[key]["frequency"]
        config = f"{key}/{freq}/{term}"

        dataset = Dataset(name=ds_name, term=term)
        # GIFT-Eval is scored on the targets alone (our reported scores), so covariate fields
        # that some datasets carry are not passed to the model.
        predictor = SimonsPredictor(model, prediction_length=dataset.prediction_length,
                                    use_covariates=False)
        result = evaluate_model(
            predictor,
            test_data=dataset.test_data,
            metrics=metrics,
            batch_size=1024,
            axis=None,
            mask_invalid_label=True,
            allow_nan_forecast=False,
            seasonality=get_seasonality(dataset.freq),
        )
        row = result.iloc[0]
        with open(csv_path, "a", newline="") as stream:
            csv.writer(stream).writerow([
                config, NAME,
                row["MSE[mean]"], row["MSE[0.5]"], row["MAE[0.5]"], row["MASE[0.5]"],
                row["MAPE[0.5]"], row["sMAPE[0.5]"], row["MSIS"], row["RMSE[mean]"],
                row["NRMSE[mean]"], row["ND[0.5]"], row["mean_weighted_sum_quantile_loss"],
                properties[key]["domain"], properties[key]["num_variates"],
            ])
        print(f"{config}: MASE {row['MASE[0.5]']:.4f}, "
              f"CRPS {row['mean_weighted_sum_quantile_loss']:.4f}", flush=True)

print(f"Results in {csv_path}")

In [ ]:
import numpy as np
import pandas as pd

ours = pd.read_csv(csv_path).set_index("dataset")
naive_csv = next(p for p in (REPO / "results/Seasonal_Naive/all_results.csv",
                             REPO / "results/seasonal_naive/all_results.csv") if p.exists())
naive = pd.read_csv(naive_csv).set_index("dataset").loc[ours.index]
for metric, label in (("eval_metrics/mean_weighted_sum_quantile_loss", "CRPS"),
                      ("eval_metrics/MASE[0.5]", "MASE")):
    relative = np.exp(np.mean(np.log(ours[metric] / naive[metric])))
    print(f"relative {label}: {relative:.4f}")